# Sistema RAG - Demostración Completa
## Retrieval-Augmented Generation para Q&A de Papers Científicos

**Entrevista Técnica - Take Home Project**

In [38]:
# Celda 1: Configuración de rutas y procesamiento de PDFs
import os
import json
import pandas as pd
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

print("🚀 Sistema RAG con PDFs Personalizados")
print("=" * 50)

# CORREGIR RUTA: Navegar al directorio padre si estamos en notebooks
if 'notebooks' in os.getcwd():
    os.chdir('..')
    print(f"📂 Cambiado al directorio padre: {os.getcwd()}")

# Verificar estructura de carpetas
print("📁 Estructura actual:")
for item in ['data', 'notebooks', 'pdfs', 'venv_rag']:
    if os.path.exists(item):
        print(f"   ✅ {item}/")
    else:
        print(f"   ❌ {item}/ - NO EXISTE")

# Crear carpeta pdfs si no existe
pdfs_folder = 'pdfs'
if not os.path.exists(pdfs_folder):
    os.makedirs(pdfs_folder)
    print(f"✅ Carpeta {pdfs_folder} creada")

# Verificar PDFs
pdf_files = list(Path(pdfs_folder).glob("*.pdf"))
print(f"\n📄 PDFs encontrados: {len(pdf_files)}")

if pdf_files:
    print("📚 Lista de PDFs:")
    for i, pdf in enumerate(pdf_files, 1):
        size_mb = pdf.stat().st_size / (1024*1024)
        print(f"   {i}. {pdf.name} ({size_mb:.1f} MB)")
else:
    print("⚠️ No se encontraron PDFs en la carpeta")
    print(f"📍 Ruta completa: {Path(pdfs_folder).absolute()}")
    print("💡 Copia tus PDFs a esta carpeta y vuelve a ejecutar")

🚀 Sistema RAG con PDFs Personalizados
📁 Estructura actual:
   ✅ data/
   ✅ notebooks/
   ✅ pdfs/
   ✅ venv_rag/

📄 PDFs encontrados: 6
📚 Lista de PDFs:
   1. Refrigerantes.pdf (0.1 MB)
   2. Carga Térmica.pdf (0.1 MB)
   3. Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia.pdf (0.7 MB)
   4. Grupo 1_Lab 3_Abril, Daza, Lopera, Murcia.pdf (0.8 MB)
   5. Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia.pdf (0.8 MB)
   6. Informe Salida Posco.pdf (1.6 MB)


In [49]:
# Celda 2: Procesador de PDFs robusto
# Instalar dependencias si no están
import subprocess
import sys

def install_if_missing(package):
    try:
        __import__(package.split('[')[0])  # Manejar paquetes con extras como 'package[extra]'
    except ImportError:
        print(f"📦 Instalando {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Instalar dependencias necesarias
for package in ['PyPDF2', 'pdfplumber']:
    install_if_missing(package)

import PyPDF2
import pdfplumber

def extract_text_from_pdf(pdf_path):
    """Extraer texto de PDF con múltiples métodos de respaldo"""
    text = ""
    filename = os.path.basename(pdf_path)
    
    # Método 1: pdfplumber (mejor para la mayoría de casos)
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                page_text = page.extract_text()
                if page_text:
                    text += f"\n--- Página {page_num} ---\n"
                    text += page_text
                    
        if text.strip():
            print(f"✅ {filename}: {len(text)} caracteres extraídos (pdfplumber)")
            return text.strip()
    except Exception as e:
        print(f"⚠️ pdfplumber falló para {filename}: {e}")
    
    # Método 2: PyPDF2 (respaldo)
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page_num, page in enumerate(pdf_reader.pages, 1):
                page_text = page.extract_text()
                if page_text:
                    text += f"\n--- Página {page_num} ---\n"
                    text += page_text
                    
        if text.strip():
            print(f"✅ {filename}: {len(text)} caracteres extraídos (PyPDF2)")
            return text.strip()
    except Exception as e:
        print(f"❌ PyPDF2 también falló para {filename}: {e}")
    
    return ""

def split_text_into_chunks(text, filename, chunk_size=1500, overlap=200):
    """Dividir texto en chunks manejables"""
    
    if len(text) <= chunk_size:
        return [{
            'id': f"{filename}_chunk_1",
            'title': f"{filename}",
            'content': text,
            'source': filename,
            'chunk_id': 1,
            'total_chunks': 1
        }]
    
    chunks = []
    start = 0
    chunk_num = 1
    
    while start < len(text):
        end = min(start + chunk_size, len(text))
        
        # Buscar punto de corte natural
        if end < len(text):
            for i in range(end, max(start + chunk_size//2, end - 200), -1):
                if text[i] in '.\n!?':
                    end = i + 1
                    break
        
        chunk_text = text[start:end].strip()
        
        if chunk_text:
            chunks.append({
                'id': f"{filename}_chunk_{chunk_num}",
                'title': f"{filename} (Parte {chunk_num})",
                'content': chunk_text,
                'source': filename,
                'chunk_id': chunk_num,
                'total_chunks': 0  # Se actualizará después
            })
            chunk_num += 1
        
        start = end - overlap if end < len(text) else end
    
    # Actualizar total_chunks
    for chunk in chunks:
        chunk['total_chunks'] = len(chunks)
    
    print(f"📑 {filename}: dividido en {len(chunks)} chunks")
    return chunks

# Procesar todos los PDFs
def process_all_pdfs():
    """Procesar todos los PDFs en la carpeta"""
    
    pdf_files = list(Path('pdfs').glob("*.pdf"))
    
    if not pdf_files:
        print("❌ No se encontraron PDFs")
        return []
    
    all_documents = []
    
    for pdf_path in pdf_files:
        filename = pdf_path.stem
        print(f"\n🔄 Procesando: {pdf_path.name}")
        
        # Extraer texto
        full_text = extract_text_from_pdf(pdf_path)
        
        if full_text and len(full_text) > 50:  # Texto válido
            chunks = split_text_into_chunks(full_text, filename)
            all_documents.extend(chunks)
        else:
            print(f"❌ No se pudo extraer texto útil de {filename}")
    
    return all_documents

# Ejecutar procesamiento
print("📄 Procesando PDFs...")
pdf_documents = process_all_pdfs()

if pdf_documents:
    pdf_df = pd.DataFrame(pdf_documents)
    print(f"\n✅ Dataset creado: {len(pdf_df)} documentos")
    
    # Estadísticas
    print(f"📊 Estadísticas:")
    print(f"   📄 Total chunks: {len(pdf_df)}")
    print(f"   📚 Fuentes únicas: {pdf_df['source'].nunique()}")
    print(f"   📝 Promedio caracteres: {pdf_df['content'].str.len().mean():.0f}")
    
    # Mostrar muestra
    print(f"\n🔍 Primeros documentos:")
    for i, row in pdf_df.head(3).iterrows():
        print(f"   {i+1}. {row['title']}")
        print(f"      📄 Fuente: {row['source']}")
        print(f"      📏 Tamaño: {len(row['content'])} chars")
else:
    print("❌ No se pudieron procesar PDFs")
    print("💡 Verifica que los PDFs no estén corruptos o protegidos")

📄 Procesando PDFs...

🔄 Procesando: Refrigerantes.pdf
✅ Refrigerantes.pdf: 9161 caracteres extraídos (pdfplumber)
📑 Refrigerantes: dividido en 8 chunks

🔄 Procesando: Carga Térmica.pdf
✅ Carga Térmica.pdf: 6116 caracteres extraídos (pdfplumber)
📑 Carga Térmica: dividido en 5 chunks

🔄 Procesando: Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia.pdf


Cannot set gray non-stroke color because /'P97' is an invalid float value
Cannot set gray non-stroke color because /'P112' is an invalid float value
Cannot set gray non-stroke color because /'P117' is an invalid float value
Cannot set gray non-stroke color because /'P124' is an invalid float value
Cannot set gray non-stroke color because /'P128' is an invalid float value
Cannot set gray non-stroke color because /'P138' is an invalid float value
Cannot set gray non-stroke color because /'P153' is an invalid float value
Cannot set gray non-stroke color because /'P158' is an invalid float value
Cannot set gray non-stroke color because /'P162' is an invalid float value
Cannot set gray non-stroke color because /'P167' is an invalid float value
Cannot set gray non-stroke color because /'P172' is an invalid float value
Cannot set gray non-stroke color because /'P177' is an invalid float value


✅ Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia.pdf: 33911 caracteres extraídos (pdfplumber)
📑 Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia: dividido en 27 chunks

🔄 Procesando: Grupo 1_Lab 3_Abril, Daza, Lopera, Murcia.pdf
✅ Grupo 1_Lab 3_Abril, Daza, Lopera, Murcia.pdf: 27060 caracteres extraídos (pdfplumber)
📑 Grupo 1_Lab 3_Abril, Daza, Lopera, Murcia: dividido en 21 chunks

🔄 Procesando: Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia.pdf
✅ Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia.pdf: 41595 caracteres extraídos (pdfplumber)
📑 Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia: dividido en 33 chunks

🔄 Procesando: Informe Salida Posco.pdf
✅ Informe Salida Posco.pdf: 51448 caracteres extraídos (pdfplumber)
📑 Informe Salida Posco: dividido en 41 chunks

✅ Dataset creado: 135 documentos
📊 Estadísticas:
   📄 Total chunks: 135
   📚 Fuentes únicas: 6
   📝 Promedio caracteres: 1444

🔍 Primeros documentos:
   1. Refrigerantes (Parte 1)
      📄 Fuente: Refrigerantes
      📏 Tamaño: 1414 chars
   2. Refrigerante

In [11]:
# Celda 3: Configurar sistema RAG para PDFs
print("🧠 Configurando sistema RAG para PDFs...")

# Verificar que tenemos documentos
if 'pdf_df' not in globals() or pdf_df.empty:
    print("❌ No hay documentos PDF procesados")
    print("🔄 Ejecuta la celda anterior primero")
else:
    print(f"✅ Documentos disponibles: {len(pdf_df)}")

# Imports RAG
import chromadb
from sentence_transformers import SentenceTransformer
import time

# Configurar ChromaDB
client = chromadb.Client()

try:
    client.delete_collection("pdf_documents")
    print("🧹 Colección anterior eliminada")
except:
    pass

collection = client.create_collection("pdf_documents")
print("✅ Nueva base de datos vectorial configurada")

# Cargar modelo de embeddings
print("🧠 Cargando modelo de embeddings...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Modelo de embeddings cargado")

# Preparar documentos para embeddings
documents = []
metadatas = []
ids = []

print("📋 Preparando documentos para vectorización...")

for _, doc in pdf_df.iterrows():
    # Combinar título y contenido para mejor contexto
    doc_text = f"Documento: {doc['title']}\n\nContenido:\n{doc['content']}"
    
    documents.append(doc_text)
    metadatas.append({
        "title": doc["title"],
        "source": doc["source"],
        "chunk_id": int(doc["chunk_id"]),
        "total_chunks": int(doc["total_chunks"]),
        "content_length": len(doc["content"])
    })
    ids.append(doc["id"])

print(f"📄 {len(documents)} documentos preparados para embeddings")

# Generar embeddings
print("🔄 Generando embeddings (esto puede tomar 1-2 minutos)...")
embeddings = embedding_model.encode(documents, show_progress_bar=True)
print(f"✅ Embeddings generados: {embeddings.shape}")

# Almacenar en ChromaDB
print("💾 Almacenando en base de datos vectorial...")
collection.add(
    embeddings=embeddings.tolist(),
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"✅ {collection.count()} documentos almacenados en base de datos")
print("🎯 Sistema RAG configurado y listo")

🧠 Configurando sistema RAG para PDFs...
✅ Documentos disponibles: 81
🧹 Colección anterior eliminada
✅ Nueva base de datos vectorial configurada
🧠 Cargando modelo de embeddings...
✅ Modelo de embeddings cargado
📋 Preparando documentos para vectorización...
📄 81 documentos preparados para embeddings
🔄 Generando embeddings (esto puede tomar 1-2 minutos)...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

✅ Embeddings generados: (81, 384)
💾 Almacenando en base de datos vectorial...
✅ 81 documentos almacenados en base de datos
🎯 Sistema RAG configurado y listo


In [50]:
# Celda 4: Sistema RAG especializado para PDFs
print("🔧 Definiendo sistema RAG para PDFs...")

class PDFRAGSystem:
    def __init__(self, collection, embedding_model):
        self.collection = collection
        self.embedding_model = embedding_model
        print("✅ PDFRAGSystem inicializado")
    
    def query(self, question, n_results=3):
        """Realizar consulta completa en documentos PDF"""
        print(f"🔍 Consulta: '{question}'")
        
        start_time = time.time()
        
        # Generar embedding de la consulta
        query_embedding = self.embedding_model.encode([question])
        
        # Buscar documentos similares
        results = self.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=n_results,
            include=["documents", "metadatas", "distances"]
        )
        
        search_time = time.time() - start_time
        
        # Mostrar resultados detallados
        print(f"\n📚 Documentos encontrados ({search_time*1000:.1f}ms):")
        
        for i, (meta, distance) in enumerate(zip(results['metadatas'][0], results['distances'][0])):
            relevance = 1 - distance
            print(f"\n  {i+1}. {meta['title']}")
            print(f"     📂 Fuente: {meta['source']}")
            print(f"     📊 Relevancia: {relevance:.3f}")
            print(f"     📏 Longitud: {meta['content_length']} caracteres")
            
            if meta['total_chunks'] > 1:
                print(f"     📑 Sección {meta['chunk_id']} de {meta['total_chunks']}")
        
        # Generar respuesta contextual
        response = self.generate_contextual_response(question, results)
        print(f"\n🤖 Respuesta generada:")
        print(response)
        
        return results
    
    def generate_contextual_response(self, question, results):
        """Generar respuesta contextual basada en PDFs"""
        if not results['metadatas'][0]:
            return "No encontré información relevante en tus documentos PDF."
        
        # Analizar resultados
        sources = [meta['source'] for meta in results['metadatas'][0]]
        titles = [meta['title'] for meta in results['metadatas'][0]]
        relevances = [1 - dist for dist in results['distances'][0]]
        
        # Construir respuesta
        best_relevance = max(relevances)
        unique_sources = list(set(sources))
        
        if best_relevance > 0.7:
            confidence = "alta"
        elif best_relevance > 0.5:
            confidence = "media"
        else:
            confidence = "baja"
        
        response = f"Basado en {len(results['metadatas'][0])} secciones de tus documentos PDF (confianza {confidence}), "
        
        if len(unique_sources) == 1:
            response += f"encontré información relevante en '{unique_sources[0]}'. "
        else:
            response += f"encontré información en {len(unique_sources)} documentos: {', '.join(unique_sources)}. "
        
        response += f"La sección más relevante es '{titles[0]}' con una puntuación de {best_relevance:.3f}. "
        
        # Agregar contexto específico basado en la pregunta
        question_lower = question.lower()
        if any(word in question_lower for word in ['qué es', 'define', 'definición', 'concepto']):
            response += "Los documentos contienen definiciones y explicaciones conceptuales sobre el tema."
        elif any(word in question_lower for word in ['cómo', 'pasos', 'proceso', 'método']):
            response += "Los documentos describen procesos y metodologías relacionadas."
        elif any(word in question_lower for word in ['ventajas', 'beneficios', 'aplicaciones']):
            response += "Los documentos incluyen información sobre aplicaciones y beneficios."
        elif any(word in question_lower for word in ['problemas', 'desafíos', 'limitaciones']):
            response += "Los documentos discuten desafíos y limitaciones en el área."
        else:
            response += "Los documentos contienen información detallada que responde a tu consulta."
        
        return response
    
    def get_statistics(self):
        """Obtener estadísticas del sistema"""
        count = self.collection.count()
        
        # Obtener metadatos de todos los documentos
        all_results = self.collection.get(include=["metadatas"])
        sources = [meta['source'] for meta in all_results['metadatas']]
        
        return {
            'total_documents': count,
            'unique_sources': len(set(sources)),
            'sources': list(set(sources))
        }

# Inicializar sistema RAG para PDFs
pdf_rag_system = PDFRAGSystem(collection, embedding_model)

# Mostrar estadísticas
stats = pdf_rag_system.get_statistics()
print(f"\n📊 Estadísticas del sistema:")
print(f"   📄 Total documentos: {stats['total_documents']}")
print(f"   📚 Fuentes únicas: {stats['unique_sources']}")
print(f"   📋 Fuentes: {', '.join(stats['sources'])}")

print("\n✅ Sistema RAG para PDFs listo para usar")

🔧 Definiendo sistema RAG para PDFs...
✅ PDFRAGSystem inicializado

📊 Estadísticas del sistema:
   📄 Total documentos: 81
   📚 Fuentes únicas: 3
   📋 Fuentes: Grupo 1_Lab 3_Abril, Daza, Lopera, Murcia, Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia, Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia

✅ Sistema RAG para PDFs listo para usar


In [13]:
# Celda 5: Demo con consultas adaptables a cualquier contenido
print("🎯 DEMO CON TUS PDFs PERSONALIZADOS")
print("=" * 50)

# Obtener información sobre los documentos disponibles
stats = pdf_rag_system.get_statistics()
sources = stats['sources']

print(f"📚 Documentos disponibles: {', '.join(sources)}")

# Consultas genéricas que funcionan con cualquier contenido
consultas_generales = [
    "¿De qué trata el documento principal?",
    "¿Cuáles son los conceptos clave mencionados?", 
    "¿Qué información importante contiene?",
    "Resumen del contenido más relevante"
]

# Consultas más específicas (adaptar según tu contenido)
consultas_especificas = [
    "¿Cuáles son las ideas principales?",
    "¿Qué metodologías se describen?",
    "¿Qué conclusiones se mencionan?",
    "¿Cuáles son los puntos más importantes?"
]

# Combinar consultas
todas_las_consultas = consultas_generales + consultas_especificas

print(f"\n🔍 Ejecutando {len(consultas_generales)} consultas de demostración:")

# Ejecutar demo
for i, consulta in enumerate(consultas_generales, 1):
    print(f"\n{'='*70}")
    print(f"CONSULTA {i}:")
    print(f"{'='*70}")
    
    try:
        resultado = pdf_rag_system.query(consulta)
        print(f"✅ Consulta procesada exitosamente")
        
        # Mostrar métricas
        relevancia_max = 1 - resultado['distances'][0][0]
        print(f"📊 Relevancia máxima: {relevancia_max:.3f}")
        
    except Exception as e:
        print(f"❌ Error en consulta: {e}")
    
    if i < len(consultas_generales):
        print("\n⏳ Procesando siguiente consulta...")

print("\n🎉 Demo básico completado!")

🎯 DEMO CON TUS PDFs PERSONALIZADOS
📚 Documentos disponibles: Grupo 1_Lab 3_Abril, Daza, Lopera, Murcia, Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia, Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia

🔍 Ejecutando 4 consultas de demostración:

CONSULTA 1:
🔍 Consulta: '¿De qué trata el documento principal?'

📚 Documentos encontrados (19.6ms):

  1. Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia (Parte 27)
     📂 Fuente: Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia
     📊 Relevancia: -0.004
     📏 Longitud: 747 caracteres
     📑 Sección 27 de 27

  2. Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia (Parte 22)
     📂 Fuente: Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia
     📊 Relevancia: -0.075
     📏 Longitud: 1457 caracteres
     📑 Sección 22 de 27

  3. Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia (Parte 10)
     📂 Fuente: Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia
     📊 Relevancia: -0.123
     📏 Longitud: 1473 caracteres
     📑 Sección 10 de 33

🤖 Respuesta generada:
Basado en 3 secciones de tus documentos PDF (con

In [51]:
# Celda 6.1: Detector de contenido para preguntas específicas
print("🔍 ANALIZADOR DE CONTENIDO DE TUS PDFs")
print("=" * 45)

def analizar_contenido():
    """Analizar qué tipo de contenido hay en los PDFs"""
    
    # Obtener una muestra del contenido
    sample_results = collection.get(
        limit=5,
        include=["documents", "metadatas"]
    )
    
    # Analizar palabras clave comunes
    todo_el_texto = " ".join(sample_results['documents']).lower()
    
    # Categorías de contenido
    categorias = {
        "técnico": ["algoritmo", "método", "técnica", "sistema", "proceso", "implementación"],
        "académico": ["investigación", "estudio", "análisis", "conclusión", "resultado", "hipótesis"],
        "negocio": ["empresa", "mercado", "estrategia", "cliente", "producto", "servicio"],
        "educativo": ["aprendizaje", "enseñanza", "educación", "estudiante", "curso", "lección"],
        "científico": ["experimento", "datos", "muestra", "variable", "medición", "observación"]
    }
    
    print("📊 ANÁLISIS DE CONTENIDO:")
    contenido_detectado = []
    
    for categoria, palabras in categorias.items():
        coincidencias = sum(1 for palabra in palabras if palabra in todo_el_texto)
        porcentaje = (coincidencias / len(palabras)) * 100
        
        if porcentaje > 20:  # Si encuentra más del 20% de las palabras
            contenido_detectado.append(categoria)
            print(f"   ✅ {categoria.upper()}: {porcentaje:.0f}% coincidencia")
        else:
            print(f"   ⚪ {categoria}: {porcentaje:.0f}% coincidencia")
    
    # Sugerir preguntas basadas en el contenido detectado
    print(f"\n💡 PREGUNTAS SUGERIDAS PARA TU CONTENIDO:")
    
    if "técnico" in contenido_detectado:
        print("   🔧 ¿Qué métodos o técnicas se describen?")
        print("   🔧 ¿Cómo funciona el sistema mencionado?")
        print("   🔧 ¿Qué procesos de implementación se explican?")
    
    if "académico" in contenido_detectado:
        print("   📚 ¿Cuáles son los resultados de la investigación?")
        print("   📚 ¿Qué conclusiones se presentan?")
        print("   📚 ¿Qué metodología de estudio se utilizó?")
    
    if "negocio" in contenido_detectado:
        print("   💼 ¿Cuál es la estrategia de negocio propuesta?")
        print("   💼 ¿Qué oportunidades de mercado se identifican?")
        print("   💼 ¿Cómo se puede mejorar el servicio al cliente?")
    
    if "educativo" in contenido_detectado:
        print("   📖 ¿Qué métodos de enseñanza se recomiendan?")
        print("   📖 ¿Cómo mejorar el proceso de aprendizaje?")
        print("   📖 ¿Qué recursos educativos se mencionan?")
    
    if "científico" in contenido_detectado:
        print("   🔬 ¿Qué experimentos se realizaron?")
        print("   🔬 ¿Cuáles fueron los datos obtenidos?")
        print("   🔬 ¿Qué variables se estudiaron?")
    
    if not contenido_detectado:
        print("   ❓ ¿Cuál es el tema principal del documento?")
        print("   ❓ ¿Qué información importante contiene?")
        print("   ❓ ¿Cuáles son los puntos clave mencionados?")
    
    return contenido_detectado

# Ejecutar análisis
tipos_contenido = analizar_contenido()

print(f"\n🎯 PRUEBA UNA PREGUNTA ESPECÍFICA:")
print("Basándote en las sugerencias de arriba, haz una pregunta:")

# PREGUNTA ESPECÍFICA BASADA EN EL ANÁLISIS:
pregunta_especifica = "¿Cuáles son los métodos principales que se describen?"

print(f"\n🔍 Pregunta específica: '{pregunta_especifica}'")
resultado_especifico = consulta_simple(pregunta_especifica)

🔍 ANALIZADOR DE CONTENIDO DE TUS PDFs
📊 ANÁLISIS DE CONTENIDO:
   ⚪ técnico: 17% coincidencia
   ⚪ académico: 17% coincidencia
   ⚪ negocio: 0% coincidencia
   ⚪ educativo: 0% coincidencia
   ✅ CIENTÍFICO: 33% coincidencia

💡 PREGUNTAS SUGERIDAS PARA TU CONTENIDO:
   🔬 ¿Qué experimentos se realizaron?
   🔬 ¿Cuáles fueron los datos obtenidos?
   🔬 ¿Qué variables se estudiaron?

🎯 PRUEBA UNA PREGUNTA ESPECÍFICA:
Basándote en las sugerencias de arriba, haz una pregunta:

🔍 Pregunta específica: '¿Cuáles son los métodos principales que se describen?'
🔍 Pregunta: '¿Cuáles son los métodos principales que se describen?'
--------------------------------------------------
📚 DOCUMENTOS ENCONTRADOS:
   1. Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia - Relevancia: 4.3%
   2. Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia - Relevancia: -2.8%
   3. Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia - Relevancia: -6.8%

🤖 RESPUESTA CLARA:
❌ La información encontrada tiene BAJA RELEVANCIA
   Nivel de confianza: 4.3%
💡 

In [43]:
# Celda 6.2: Respuestas SÚPER DIRECTAS
print("⚡ RESPUESTAS SÚPER DIRECTAS")
print("=" * 30)

def respuesta_directa(pregunta):
    """Dar respuestas muy directas y claras"""
    
    print(f"❓ PREGUNTA: {pregunta}")
    
    # Buscar
    query_embedding = embedding_model.encode([pregunta])
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=1,  # Solo el mejor resultado
        include=["documents", "metadatas", "distances"]
    )
    
    if not results['metadatas'][0]:
        print("❌ RESPUESTA: No encontré información sobre eso")
        return
    
    relevancia = (1 - results['distances'][0][0]) * 100
    documento = results['metadatas'][0][0]['source']
    
    print(f"📄 DOCUMENTO: {documento}")
    print(f"📊 CONFIANZA: {relevancia:.0f}%")
    
    if relevancia > 70:
        print("✅ RESPUESTA: SÍ, encontré información muy relevante sobre eso")
    elif relevancia > 50:
        print("👍 RESPUESTA: SÍ, hay información relacionada con eso")
    elif relevancia > 30:
        print("⚠️ RESPUESTA: Hay información PARCIAL sobre eso")
    else:
        print("❌ RESPUESTA: NO encontré información clara sobre eso")
    
    return relevancia

# PRUEBAS RÁPIDAS:
print("\n🧪 PRUEBAS RÁPIDAS:")

preguntas_rapidas = [
    "¿Hay información sobre fluencia?",
    "¿Se menciona metodología?",
    "¿Hay conclusiones importantes?",
    "¿Se habla de resultados?"
]

for pregunta in preguntas_rapidas:
    relevancia = respuesta_directa(pregunta)
    print("-" * 40)

print(f"\n🎯 TU PREGUNTA PERSONALIZADA:")
# CAMBIA ESTA PREGUNTA POR LA QUE QUIERAS:
tu_pregunta = "Cuales son los Halocarbonados del grupo I"

respuesta_directa(tu_pregunta)

print(f"\n💡 Cambia 'tu_pregunta' y ejecuta de nuevo para probar otras consultas")

⚡ RESPUESTAS SÚPER DIRECTAS

🧪 PRUEBAS RÁPIDAS:
❓ PREGUNTA: ¿Hay información sobre fluencia?
📄 DOCUMENTO: Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia
📊 CONFIANZA: 4%
❌ RESPUESTA: NO encontré información clara sobre eso
----------------------------------------
❓ PREGUNTA: ¿Se menciona metodología?
📄 DOCUMENTO: Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia
📊 CONFIANZA: 7%
❌ RESPUESTA: NO encontré información clara sobre eso
----------------------------------------
❓ PREGUNTA: ¿Hay conclusiones importantes?
📄 DOCUMENTO: Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia
📊 CONFIANZA: -10%
❌ RESPUESTA: NO encontré información clara sobre eso
----------------------------------------
❓ PREGUNTA: ¿Se habla de resultados?
📄 DOCUMENTO: Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia
📊 CONFIANZA: -1%
❌ RESPUESTA: NO encontré información clara sobre eso
----------------------------------------

🎯 TU PREGUNTA PERSONALIZADA:
❓ PREGUNTA: Cuales son los Halocarbonados del grupo I
📄 DOCUMENTO: Grupo 1_Lab 2_Abril, Daza, Lo

In [52]:
# Celda 6: Zona de pruebas SIMPLE y CLARA
print("🎮 ZONA DE PRUEBAS SIMPLE")
print("=" * 30)

def consulta_simple(pregunta):
    """Función simplificada para consultas claras"""
    print(f"🔍 Pregunta: '{pregunta}'")
    print("-" * 50)
    
    try:
        # Buscar información
        query_embedding = embedding_model.encode([pregunta])
        results = collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=3,
            include=["documents", "metadatas", "distances"]
        )
        
        if not results['metadatas'][0]:
            print("❌ No encontré información relevante")
            return
        
        # Mostrar resultados de forma simple
        print("📚 DOCUMENTOS ENCONTRADOS:")
        for i, (meta, distance) in enumerate(zip(results['metadatas'][0], results['distances'][0])):
            relevancia = (1 - distance) * 100  # Convertir a porcentaje
            print(f"   {i+1}. {meta['source']} - Relevancia: {relevancia:.1f}%")
        
        # Respuesta clara y directa
        mejor_documento = results['metadatas'][0][0]['source']
        mejor_relevancia = (1 - results['distances'][0][0]) * 100
        
        print(f"\n🤖 RESPUESTA CLARA:")
        if mejor_relevancia > 70:
            print(f"✅ Encontré información MUY RELEVANTE en '{mejor_documento}'")
            print(f"   Nivel de confianza: {mejor_relevancia:.1f}%")
        elif mejor_relevancia > 50:
            print(f"👍 Encontré información RELEVANTE en '{mejor_documento}'")
            print(f"   Nivel de confianza: {mejor_relevancia:.1f}%")
        elif mejor_relevancia > 30:
            print(f"⚠️ Encontré información PARCIAL en '{mejor_documento}'")
            print(f"   Nivel de confianza: {mejor_relevancia:.1f}%")
        else:
            print(f"❌ La información encontrada tiene BAJA RELEVANCIA")
            print(f"   Nivel de confianza: {mejor_relevancia:.1f}%")
            print("💡 Intenta hacer una pregunta más específica")
        
        # Mostrar fragmento del contenido más relevante
        if mejor_relevancia > 30:
            documento_texto = results['documents'][0][0]
            # Extraer las primeras 200 caracteres del contenido
            if "Contenido:" in documento_texto:
                contenido = documento_texto.split("Contenido:")[1][:200].strip()
                print(f"\n📄 FRAGMENTO RELEVANTE:")
                print(f"   '{contenido}...'")
        
        return results
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# EJEMPLOS DE USO SIMPLE:
print("💡 EJEMPLOS DE PREGUNTAS SIMPLES:")
ejemplos = [
    "¿De qué trata el documento?",
    "¿Cuáles son las ideas principales?", 
    "¿Qué conceptos importantes se mencionan?",
    "¿Hay información sobre metodología?",
    "¿Qué conclusiones se presentan?"
]

for i, ejemplo in enumerate(ejemplos, 1):
    print(f"   {i}. {ejemplo}")

print(f"\n" + "="*60)
print("🎯 HAZ TU PREGUNTA AQUÍ:")
print("="*60)

# CAMBIA ESTA PREGUNTA:
mi_pregunta = "¿Quienes son los autores?"

# Ejecutar consulta
resultado = consulta_simple(mi_pregunta)

print(f"\n" + "="*60)
print("💡 PRUEBA OTRAS PREGUNTAS:")
print("Cambia 'mi_pregunta' arriba y vuelve a ejecutar esta celda")
print("="*60)

🎮 ZONA DE PRUEBAS SIMPLE
💡 EJEMPLOS DE PREGUNTAS SIMPLES:
   1. ¿De qué trata el documento?
   2. ¿Cuáles son las ideas principales?
   3. ¿Qué conceptos importantes se mencionan?
   4. ¿Hay información sobre metodología?
   5. ¿Qué conclusiones se presentan?

🎯 HAZ TU PREGUNTA AQUÍ:
🔍 Pregunta: '¿Quienes son los autores?'
--------------------------------------------------
📚 DOCUMENTOS ENCONTRADOS:
   1. Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia - Relevancia: -19.3%
   2. Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia - Relevancia: -22.1%
   3. Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia - Relevancia: -23.7%

🤖 RESPUESTA CLARA:
❌ La información encontrada tiene BAJA RELEVANCIA
   Nivel de confianza: -19.3%
💡 Intenta hacer una pregunta más específica

💡 PRUEBA OTRAS PREGUNTAS:
Cambia 'mi_pregunta' arriba y vuelve a ejecutar esta celda


In [44]:
# Celda NUEVA: Sistema RAG que responde con CONTENIDO REAL
print("🎯 SISTEMA RAG CON RESPUESTAS REALES DEL PDF")
print("=" * 50)

class RAGContenidoReal:
    def __init__(self, collection, embedding_model):
        self.collection = collection
        self.embedding_model = embedding_model
    
    def responder_con_contenido(self, pregunta, mostrar_fragmentos=True):
        """Responder con contenido REAL extraído del PDF"""
        
        print(f"❓ PREGUNTA: {pregunta}")
        print("=" * 60)
        
        # Buscar documentos relevantes
        query_embedding = self.embedding_model.encode([pregunta])
        results = self.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=3,
            include=["documents", "metadatas", "distances"]
        )
        
        if not results['documents'][0]:
            print("❌ No encontré información relevante")
            return
        
        print("📚 INFORMACIÓN ENCONTRADA:")
        print("-" * 30)
        
        for i, (documento, meta, distancia) in enumerate(zip(
            results['documents'][0], 
            results['metadatas'][0], 
            results['distances'][0]
        )):
            relevancia = (1 - distancia) * 100
            
            print(f"\n📄 FUENTE {i+1}: {meta['source']}")
            print(f"📊 RELEVANCIA: {relevancia:.1f}%")
            
            if relevancia > 20:  # Solo mostrar si es algo relevante
                
                # Extraer el contenido real (después de "Contenido:")
                if "Contenido:" in documento:
                    contenido_real = documento.split("Contenido:")[1].strip()
                else:
                    contenido_real = documento
                
                # Buscar la parte más relevante del contenido
                fragmentos = self.encontrar_fragmentos_relevantes(contenido_real, pregunta)
                
                if fragmentos:
                    print(f"💬 RESPUESTA DEL DOCUMENTO:")
                    for j, fragmento in enumerate(fragmentos[:2], 1):  # Máximo 2 fragmentos
                        print(f"   {j}. \"{fragmento}\"")
                else:
                    # Si no encuentra fragmentos específicos, mostrar inicio del contenido
                    print(f"💬 CONTENIDO GENERAL:")
                    print(f"   \"{contenido_real[:300]}...\"")
                
                print("-" * 40)
        
        return results
    
    def encontrar_fragmentos_relevantes(self, contenido, pregunta, longitud_fragmento=150):
        """Encontrar fragmentos específicos que respondan a la pregunta"""
        
        # Palabras clave de la pregunta
        palabras_pregunta = pregunta.lower().split()
        palabras_importantes = [p for p in palabras_pregunta if len(p) > 3 and p not in ['cuál', 'cuáles', 'cómo', 'cuándo', 'dónde', 'quién']]
        
        # Dividir contenido en oraciones
        oraciones = contenido.replace('\n', ' ').split('.')
        fragmentos_relevantes = []
        
        for oracion in oraciones:
            oracion = oracion.strip()
            if len(oracion) < 20:  # Saltar oraciones muy cortas
                continue
                
            # Calcular relevancia de la oración
            oracion_lower = oracion.lower()
            coincidencias = sum(1 for palabra in palabras_importantes if palabra in oracion_lower)
            
            if coincidencias > 0:
                # Expandir la oración para dar contexto
                inicio = max(0, contenido.find(oracion) - 50)
                fin = min(len(contenido), contenido.find(oracion) + len(oracion) + 100)
                fragmento_expandido = contenido[inicio:fin].strip()
                
                fragmentos_relevantes.append((coincidencias, fragmento_expandido))
        
        # Ordenar por relevancia y devolver los mejores
        fragmentos_relevantes.sort(key=lambda x: x[0], reverse=True)
        return [f[1] for f in fragmentos_relevantes[:3]]
    
    def busqueda_exacta(self, termino_busqueda):
        """Buscar un término exacto en todos los documentos"""
        
        print(f"🔍 BÚSQUEDA EXACTA: '{termino_busqueda}'")
        print("=" * 50)
        
        # Obtener todos los documentos
        all_docs = self.collection.get(include=["documents", "metadatas"])
        
        resultados_encontrados = []
        
        for documento, meta in zip(all_docs['documents'], all_docs['metadatas']):
            # Extraer contenido real
            if "Contenido:" in documento:
                contenido = documento.split("Contenido:")[1]
            else:
                contenido = documento
            
            # Buscar término (insensible a mayúsculas)
            if termino_busqueda.lower() in contenido.lower():
                # Encontrar todas las ocurrencias
                contenido_lower = contenido.lower()
                termino_lower = termino_busqueda.lower()
                
                posiciones = []
                inicio = 0
                while True:
                    pos = contenido_lower.find(termino_lower, inicio)
                    if pos == -1:
                        break
                    posiciones.append(pos)
                    inicio = pos + 1
                
                resultados_encontrados.append((meta['source'], contenido, posiciones))
        
        if not resultados_encontrados:
            print(f"❌ No se encontró '{termino_busqueda}' en ningún documento")
            return
        
        print(f"✅ Encontrado en {len(resultados_encontrados)} documento(s):")
        
        for fuente, contenido, posiciones in resultados_encontrados:
            print(f"\n📄 DOCUMENTO: {fuente}")
            print(f"🎯 OCURRENCIAS: {len(posiciones)}")
            
            # Mostrar contexto alrededor de cada ocurrencia
            for i, pos in enumerate(posiciones[:3], 1):  # Máximo 3 ocurrencias por documento
                inicio_contexto = max(0, pos - 80)
                fin_contexto = min(len(contenido), pos + len(termino_busqueda) + 80)
                contexto = contenido[inicio_contexto:fin_contexto]
                
                # Resaltar el término encontrado
                contexto_resaltado = contexto.replace(
                    termino_busqueda, 
                    f"**{termino_busqueda}**"
                )
                
                print(f"   {i}. \"...{contexto_resaltado}...\"")
            
            print("-" * 40)

# Crear el sistema mejorado
rag_real = RAGContenidoReal(collection, embedding_model)

print("✅ Sistema RAG con contenido real inicializado")

🎯 SISTEMA RAG CON RESPUESTAS REALES DEL PDF
✅ Sistema RAG con contenido real inicializado


In [45]:
# Celda de PRUEBAS REALES
print("🧪 PRUEBAS CON RESPUESTAS REALES DEL PDF")
print("=" * 50)

# Ejemplo 1: Pregunta general
print("EJEMPLO 1 - Pregunta general:")
rag_real.responder_con_contenido("¿De qué trata el documento?")

print("\n" + "="*70)

# Ejemplo 2: Búsqueda exacta de un término
print("EJEMPLO 2 - Búsqueda exacta:")
rag_real.busqueda_exacta("fluencia")  # Cambia por un término que sepas que está en tu PDF

print("\n" + "="*70)

# Ejemplo 3: Tu pregunta específica
print("EJEMPLO 3 - Tu pregunta específica:")
tu_pregunta_especifica = "¿Qué es la fluencia?"  # CAMBIA ESTA PREGUNTA

rag_real.responder_con_contenido(tu_pregunta_especifica)

🧪 PRUEBAS CON RESPUESTAS REALES DEL PDF
EJEMPLO 1 - Pregunta general:
❓ PREGUNTA: ¿De qué trata el documento?
📚 INFORMACIÓN ENCONTRADA:
------------------------------

📄 FUENTE 1: Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia
📊 RELEVANCIA: 7.3%

📄 FUENTE 2: Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia
📊 RELEVANCIA: -4.2%

📄 FUENTE 3: Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia
📊 RELEVANCIA: -5.4%

EJEMPLO 2 - Búsqueda exacta:
🔍 BÚSQUEDA EXACTA: 'fluencia'
✅ Encontrado en 14 documento(s):

📄 DOCUMENTO: Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia
🎯 OCURRENCIAS: 4
   1. "...entran la resistencia mínima a tracción
(80000 psi o 550 MPa), resistencia a la **fluencia** mínima (60000 psi o 420 MPa),
resistencia a la **fluencia** máxima (78000 psi o 540..."
   2. "... MPa), resistencia a la **fluencia** mínima (60000 psi o 420 MPa),
resistencia a la **fluencia** máxima (78000 psi o 540 MPa) y un máximo de
alargamiento igual a 8 pulgadas o 2..."
   3. "...) y un máximo de
alargamiento igual a 8 pulgadas o 

{'ids': [['Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia_chunk_10',
   'Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia_chunk_27',
   'Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia_chunk_4']],
 'distances': [[0.9795370101928711, 0.9840970635414124, 1.1157258749008179]],
 'metadatas': [[{'chunk_id': 10,
    'content_length': 1473,
    'source': 'Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia',
    'title': 'Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia (Parte 10)',
    'total_chunks': 33},
   {'chunk_id': 27,
    'content_length': 747,
    'source': 'Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia',
    'title': 'Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia (Parte 27)',
    'total_chunks': 27},
   {'chunk_id': 4,
    'content_length': 1489,
    'source': 'Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia',
    'title': 'Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia (Parte 4)',
    'total_chunks': 33}]],
 'embeddings': None,
 'documents': [['Documento: Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia (Parte 10)\n\nContenido:\njusta la ve

In [53]:
# Celda para EXPLORAR el contenido real de tus PDFs
print("🔍 EXPLORADOR DE CONTENIDO REAL")
print("=" * 40)

def explorar_contenido():
    """Ver qué contenido real hay en los PDFs"""
    
    # Obtener muestra de documentos
    sample = collection.get(limit=3, include=["documents", "metadatas"])
    
    print("📄 MUESTRA DEL CONTENIDO REAL:")
    
    for i, (doc, meta) in enumerate(zip(sample['documents'], sample['metadatas']), 1):
        print(f"\n📚 DOCUMENTO {i}: {meta['source']}")
        print("-" * 30)
        
        # Extraer contenido real
        if "Contenido:" in doc:
            contenido = doc.split("Contenido:")[1].strip()
        else:
            contenido = doc
        
        # Mostrar primeras líneas del contenido real
        lineas = contenido.split('\n')[:5]  # Primeras 5 líneas
        for linea in lineas:
            if linea.strip():
                print(f"   {linea.strip()[:100]}...")
        
        print(f"\n📊 Total caracteres: {len(contenido)}")
        print("-" * 50)

def buscar_palabras_clave():
    """Buscar palabras clave en el contenido"""
    
    print("\n🔑 PALABRAS CLAVE DETECTADAS:")
    
    # Obtener todo el contenido
    all_docs = collection.get(include=["documents"])
    todo_contenido = ""
    
    for doc in all_docs['documents']:
        if "Contenido:" in doc:
            todo_contenido += doc.split("Contenido:")[1] + " "
        else:
            todo_contenido += doc + " "
    
    # Encontrar palabras más comunes (más de 4 caracteres)
    import re
    palabras = re.findall(r'\b\w{4,}\b', todo_contenido.lower())
    
    from collections import Counter
    palabras_comunes = Counter(palabras).most_common(10)
    
    print("📊 Palabras más frecuentes:")
    for palabra, frecuencia in palabras_comunes:
        print(f"   {palabra}: {frecuencia} veces")
    
    return [palabra for palabra, _ in palabras_comunes]

# Ejecutar exploración
explorar_contenido()
palabras_importantes = buscar_palabras_clave()

print(f"\n💡 PRUEBA BÚSQUEDAS EXACTAS CON ESTAS PALABRAS:")
for palabra in palabras_importantes[:5]:
    print(f"   rag_real.busqueda_exacta('{palabra}')")

🔍 EXPLORADOR DE CONTENIDO REAL
📄 MUESTRA DEL CONTENIDO REAL:

📚 DOCUMENTO 1: Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia
------------------------------
   --- Página 1 ---...
   LABORATORIO 1...
   COMPORTAMIENTO A TENSIÓN EN PROBETAS DE ACERO...
   JHONATAN STIVEN MURCIA BELTRAN...
   JUAN PABLO ABRIL GUTIERREZ...

📊 Total caracteres: 1501
--------------------------------------------------

📚 DOCUMENTO 2: Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia
------------------------------
   justa la velocidad a la que la...
   máquina realizará el esfuerzo, teniendo en cuenta que esta velocidad siempre tiene...
   que mantenerse constante....
   Ilustración 3 - Velocidad de Aplicación....
   • Método de Agarre...

📊 Total caracteres: 1473
--------------------------------------------------

📚 DOCUMENTO 3: Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia
------------------------------
   RTAMENTO INGENIERÍA CIVIL Y AGRÍCOLA...
   bien definida. Adicional a esto, la norma establece que cuando un ensayo fall

In [48]:
# Celda INTERACTIVA FINAL - Con respuestas exactas
print("🎮 ZONA INTERACTIVA CON RESPUESTAS EXACTAS")
print("=" * 50)

# Función para respuestas súper específicas
def pregunta_directa(pregunta):
    print(f"\n{'='*60}")
    print(f"❓ TU PREGUNTA: {pregunta}")
    print(f"{'='*60}")
    
    # Respuesta con contenido real
    rag_real.responder_con_contenido(pregunta)
    
    print(f"\n🔍 BÚSQUEDA ADICIONAL:")
    # Extraer palabras clave de la pregunta para búsqueda exacta
    palabras = pregunta.split()
    for palabra in palabras:
        if len(palabra) > 4 and palabra.lower() not in ['cuál', 'cuáles', 'cómo', 'cuándo', 'dónde']:
            print(f"\n📍 Buscando término exacto: '{palabra}'")
            rag_real.busqueda_exacta(palabra)
            break  # Solo buscar la primera palabra relevante

# PRUEBA TUS PREGUNTAS ESPECÍFICAS:
print("🎯 HAZ PREGUNTAS ESPECÍFICAS SOBRE TU PDF:")

# CAMBIA ESTAS PREGUNTAS:
mis_preguntas = [
    "¿Qué son los refrigerantes?",
    "¿El amoniaco es considerado un gas inflamable?",
    "¿EL CO2 es un refrigerante usado desde cuando?"
]

for pregunta in mis_preguntas:
    pregunta_directa(pregunta)

print(f"\n💡 MODIFICA 'mis_preguntas' arriba con preguntas específicas sobre tu contenido")

🎮 ZONA INTERACTIVA CON RESPUESTAS EXACTAS
🎯 HAZ PREGUNTAS ESPECÍFICAS SOBRE TU PDF:

❓ TU PREGUNTA: ¿Qué son los refrigerantes?
❓ PREGUNTA: ¿Qué son los refrigerantes?
📚 INFORMACIÓN ENCONTRADA:
------------------------------

📄 FUENTE 1: Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia
📊 RELEVANCIA: -29.9%

📄 FUENTE 2: Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia
📊 RELEVANCIA: -29.9%

📄 FUENTE 3: Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia
📊 RELEVANCIA: -30.4%

🔍 BÚSQUEDA ADICIONAL:

📍 Buscando término exacto: 'refrigerantes?'
🔍 BÚSQUEDA EXACTA: 'refrigerantes?'
❌ No se encontró 'refrigerantes?' en ningún documento

❓ TU PREGUNTA: ¿El amoniaco es considerado un gas inflamable?
❓ PREGUNTA: ¿El amoniaco es considerado un gas inflamable?
📚 INFORMACIÓN ENCONTRADA:
------------------------------

📄 FUENTE 1: Grupo 1_Lab 3_Abril, Daza, Lopera, Murcia
📊 RELEVANCIA: -0.3%

📄 FUENTE 2: Grupo 1_Lab 1_Abril, Daza, Lopera, Murcia
📊 RELEVANCIA: -4.0%

📄 FUENTE 3: Grupo 1_Lab 2_Abril, Daza, Lopera, Murcia
📊 REL